# LLM Safety Evaluation Framework

This notebook demonstrates the end-to-end safety evaluation of multiple LLMs across various risk categories. We run adversarial and safety-relevant prompts through the models, score their responses using heuristic-based scorers, and visualize the results.


## 1. Setup and Configuration

We start by loading the project configuration and setting up logging.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from utils.config_loader import load_config, setup_logging, load_env_file
from evaluator import EvaluationPipeline
from evaluator import ResultsAggregator
from evaluator import run_visualizations

# Load .env if present
load_env_file()

# Load config
config = load_config()
setup_logging(config)

print(f"Project: {config['project']['name']} v{config['project']['version']}")


## 2. Run Evaluation Pipeline

The `EvaluationPipeline` handles:
- **Dynamic Discovery**: (Optional) Discovering models on local LM Studio servers.
- **LangGraph Orchestration**: Robust state machine for model calls and retries.
- **Structured Storage**: Saving results to SQLite (`data/eval_results.db`) for long-term analytics.
- **Scoring**: Running heuristic scorers across 10 safety categories.


In [ ]:
# Set discover_local=True if you have LM Studio running
pipeline = EvaluationPipeline(discover_local=False) 
pipeline_state = pipeline.run()

print(f"Processed {pipeline_state['processed_count']} model-prompt pairs for run: {pipeline_state['run_id']}")


## 3. Aggregate Results

We aggregate the per-prompt scores from SQLite into summary tables. 
The aggregator computes:
- **Overall Safety Score**: Weighted average per model.
- **Latency Stats**: Mean, median, and p95 response times.
- **Narrative Summaries**: Plain-English interpretation of model behavior.


In [ ]:
aggregator = ResultsAggregator()
summaries = aggregator.run()

overall_df = summaries['model_overall']
print("Overall Safety Scores (Framed: Target vs Baseline):")
display(overall_df[["model_name", "overall_safety_score", "mean_latency_ms", "narrative_summary"]])


## 4. Visualizations

We generate charts to compare model performance across different safety dimensions.


In [ ]:
run_visualizations()

# Display some of the generated figures
from IPython.display import Image, display as display_img

figures_dir = Path(config['paths']['figures_dir'])
display_img(Image(filename=figures_dir / "overall_safety.png"))
display_img(Image(filename=figures_dir / "latency_comparison.png"))
display_img(Image(filename=figures_dir / "category_heatmap.png"))


## 5. Detailed Category Analysis

Let's look at how models performed in specific categories.


In [ ]:
category_df = summaries['category_summary']
pivot_summary = category_df.pivot(index="model_name", columns="category", values="safe_unsafe")
print("Safety Score by Category (safe_unsafe metric):")
display(pivot_summary)


## Interpretation

- **Overall Safety Score**: A weighted average reflecting the model's robustness against the defined risks.
- **Hallucination Rate**: High scores here indicate the model confidently claimed fabricated facts as true.
- **Prompt Injection/Jailbreak Success**: Indicates the model followed adversarial instructions rather than its system prompt.

### Limitations
These results use keyword-based heuristics for scoring. While fast and reproducible, they may miss nuanced failures that an LLM-as-judge or human evaluator would catch.
